# House Price Regression

EDA, feature engineering, preprocessing, model comparison, residual analysis, model saving, and prediction inference.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from preprocessing import FeatureEngineer

## Load Dataset

In [ ]:
df = pd.read_csv("../data/house_prices.csv")
print(df.shape)
display(df.head())
display(df.info())
display(df.isnull().sum().sort_values(ascending=False))

## Clean Target

In [ ]:
df = df.drop_duplicates().copy()
df = df[df["price"] > 0].copy()
print("Cleaned shape:", df.shape)

## EDA and Visualizations

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["price"], bins=50)
plt.title("House Price Distribution")
plt.xlabel("Price")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8,5))
plt.hist(np.log1p(df["price"]), bins=50)
plt.title("Log-Transformed Price Distribution")
plt.xlabel("log1p(Price)")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["sqft_living"], df["price"], alpha=0.4, s=10)
plt.title("Price vs Living Area")
plt.xlabel("sqft_living")
plt.ylabel("Price")
plt.show()

In [ ]:
corr_cols = ["price","bedrooms","bathrooms","sqft_living","sqft_lot","floors","waterfront","view","condition","sqft_above","sqft_basement","yr_built","yr_renovated"]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10,8))
im = plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Preprocessing Pipeline

Missing values are imputed, skewed area features are log-transformed, numeric features are scaled, and categorical features are one-hot encoded.

In [ ]:
X = df.drop(columns=["price"])
y = df["price"].astype(float)
y_log = np.log1p(y)

X_fe = FeatureEngineer().fit_transform(X)
num_cols = X_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_fe.select_dtypes(exclude=[np.number]).columns.tolist()
log_cols = [c for c in ["sqft_living", "sqft_lot", "sqft_above", "sqft_basement"] if c in num_cols]
normal_num_cols = [c for c in num_cols if c not in log_cols]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("log_numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("scaler", StandardScaler())]), log_cols),
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), normal_num_cols),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)]), cat_cols),
])
print("Log-transformed feature columns:", log_cols)

## Train and Compare Models

In [ ]:
X_train, X_test, y_train_log, y_test_log, y_train, y_test = train_test_split(X, y_log, y, test_size=0.2, random_state=42)
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=80, random_state=42, min_samples_leaf=2, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=120, learning_rate=0.05, max_depth=3, random_state=42),
}
rows, fitted_models = [], {}
for name, model in models.items():
    pipe = Pipeline([("features", FeatureEngineer()), ("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train_log)
    pred = np.expm1(pipe.predict(X_test))
    rows.append({"Model": name, "RMSE": np.sqrt(mean_squared_error(y_test, pred)), "MAE": mean_absolute_error(y_test, pred), "R2": r2_score(y_test, pred)})
    fitted_models[name] = pipe
metrics_df = pd.DataFrame(rows).sort_values("RMSE")
display(metrics_df)

## Residual Analysis

In [ ]:
best_name = metrics_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = np.expm1(best_model.predict(X_test))
residuals = y_test - best_pred
print("Best model:", best_name)

plt.figure(figsize=(8,5))
plt.bar(metrics_df["Model"], metrics_df["RMSE"])
plt.title("Model Comparison by RMSE")
plt.ylabel("RMSE")
plt.xticks(rotation=20, ha="right")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(best_pred, residuals, alpha=0.5, s=14)
plt.axhline(0, linestyle="--")
plt.title(f"Residual Plot - {best_name}")
plt.xlabel("Predicted Price")
plt.ylabel("Residual")
plt.show()

plt.figure(figsize=(7,7))
plt.scatter(y_test, best_pred, alpha=0.45, s=14)
minv, maxv = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
plt.plot([minv,maxv],[minv,maxv], linestyle="--")
plt.title(f"Actual vs Predicted - {best_name}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()

## Save Best Model

In [ ]:
Path("../models").mkdir(exist_ok=True)
joblib.dump(best_model, "../models/best_house_price_model.joblib")
with open("../models/best_house_price_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
metrics_df.to_csv("../outputs/model_comparison_metrics.csv", index=False)
print("Saved best model:", best_name)

## Example Inference Cell

In [ ]:
loaded_model = joblib.load("../models/best_house_price_model.joblib")
sample_house = pd.DataFrame([{
    "date": "2014-05-02 00:00:00", "bedrooms": 3, "bathrooms": 2.0,
    "sqft_living": 1800, "sqft_lot": 5000, "floors": 1.0,
    "waterfront": 0, "view": 0, "condition": 3, "sqft_above": 1600,
    "sqft_basement": 200, "yr_built": 1995, "yr_renovated": 0,
    "street": "Example Street", "city": "Seattle", "statezip": "WA 98133", "country": "USA"
}])
predicted_price = np.expm1(loaded_model.predict(sample_house))[0]
print(f"Predicted house price: ${predicted_price:,.2f}")